# Group Project Team 009

## Introduction

provide some relevant background information on the topic so that someone unfamiliar with it will be prepared to understand the rest of your report
clearly state the question you tried to answer with your project
identify and fully describe the dataset that was used to answer the question

## Methods & Results

describe the methods you used to perform your analysis from beginning to end that narrates the analysis code.
your report should include code which:
loads data 
wrangles and cleans the data to the format necessary for the planned analysis
performs a summary of the data set that is relevant for exploratory data analysis related to the planned analysis 
creates a visualization of the dataset that is relevant for exploratory data analysis related to the planned analysis
performs the data analysis
creates a visualization of the analysis 
note: all figures should have a figure number(title) and a legend(readable)

In [ ]:
install.packages("caret")   
library(caret)
library(tidyverse)
library(tidymodels)
library(repr)
library(kknn)
players <- read_csv("individual/players.csv")

clean_players <- players |> select(c(1,2,4,6,7))|> 
filter(!is.na(Age)) |> 
mutate(subscribe = as.factor(subscribe))|>  
mutate(gender = fct_collapse(
    gender,
    Male = c("Male"),
    Female = c("Female"),
    Other = c("Non-binary", "Other", "Two-Spirited", "Prefer not to say","Agender")))

head(clean_players)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8)
ggplot(clean_players, aes(x = played_hours, fill = subscribe)) +
geom_histogram(position = "dodge", bins=5) +
labs(title = "Played Hours by Subscription",
       x = "Played hours", y = "Count of Players", fill = "Subscribe") +
theme(text = element_text(size=16)) + scale_fill_brewer(palette = 'Set2')

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8)
ggplot(clean_players, aes(x = Age, fill = subscribe)) +
  geom_histogram(position = "dodge", bins = 5) +
  labs(title = "Age Distribution by Subscription Status",
       x = "Age (years)", y = "Count of Players", fill = "Subscribe") +
  theme_minimal() +theme(text = element_text(size=16)) + scale_fill_brewer(palette = 'Set1')

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8)
ggplot(clean_players, aes(x = gender, fill = subscribe)) +
  geom_bar(position = "dodge") +
  labs(title = "Gender by Subscription Status",
       x = "Gender", y = "Count of Players", fill = "Subscribe", caption = "Figure") +
  theme_minimal() + theme(text = element_text(size=16)) + scale_fill_brewer(palette = 'Accent' ) 

## Gender(one hot encoding)

In [ ]:
clean_players_gender <- clean_players|> mutate(gender = as.factor(gender)) #one hot encoding requires it to be in factor form
gender_matrix <- model.matrix(~ gender-1, data = clean_players) # ~gender -1 means I change gender into numeric form
gender_players <- clean_players |>
  bind_cols(as_tibble(gender_matrix)) # change from matrix form to table form
head(gender_players)

In [ ]:
set.seed(1123) # set. seed function makes sure you will obtain the exact same sequence of "random" numbers every time you run the code, the number inside is arbitary
# split
gender_split <- initial_split(gender_players, prop = 0.75, strata = subscribe)
gender_train <- training(gender_split)
gender_test <- testing(gender_split)

# recipe
gender_recipe <- recipe(subscribe ~ genderOther + genderFemale+ genderMale, data = gender_train) |>
  step_scale(all_predictors()) |>
  step_center(all_predictors())

In [ ]:
head(gender_train)

In [ ]:
# specification
gender_spec_tune <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
  set_engine("kknn") |>
  set_mode("classification")

# k-values
gender_ks <- tibble(neighbors = 1:20)

# v-fold
gender_vfold <- vfold_cv(gender_train, v = 5, strata = subscribe)

In [ ]:
# workflow with validation
gender_fit <- workflow() |>
  add_recipe(gender_recipe) |>
  add_model(gender_spec_tune) |>
  tune_grid(resamples = gender_vfold, grid = gender_ks) 

# accuracy
gender_tune_accuracy <- gender_fit |> collect_metrics() |>
  filter(.metric == "accuracy") |>
  select(neighbors, mean) |>
  arrange(-mean)
head(gender_tune_accuracy)
# rm(list = ls())

In [ ]:
gender_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = 17) |>
  set_engine("kknn") |>
  set_mode("classification")

gender_recipe <- recipe(subscribe ~ genderOther + genderFemale+ genderMale, data = gender_train) |>
  step_scale(all_predictors()) |>
  step_center(all_predictors())

In [ ]:
# workflow with validation
gender_fit <- workflow() |>
  add_recipe(gender_recipe) |>
  add_model(gender_spec) |>
  fit(data = gender_train)

In [ ]:
gender_test_predictions <- predict(gender_fit, gender_test) |>
  bind_cols(gender_test)

In [ ]:
head(gender_test_predictions)

In [ ]:
gender_test_predictions |>
  metrics(truth = subscribe, estimate = .pred_class) |>
 filter(.metric == "accuracy")

In [ ]:
confusion_matrix_result <- confusion_matrix_result <- confusionMatrix(
  data      = gender_test_predictions$.pred_class,   
  reference = gender_test_predictions$subscribe,     
  positive  = "TRUE")
confusion_matrix_result

## Age (ordinal encoding)

In [ ]:
head(clean_players)

In [ ]:
set.seed(1123)
# split，非必须
age_split <- initial_split(clean_players, prop = 0.75, strata = subscribe) 
age_train <- training(players_split)
age_test <- testing(players_split)

In [ ]:
# recipe
age_recipe <- recipe(subscribe ~ Age, data = age_train) |>
  step_dummy(all_nominal(), -all_outcomes()) |>  # Convert categorical variables to dummy variables, ensure that it's all numeric before scaling
  step_scale(all_numeric(), -all_outcomes())

In [ ]:
# specification
age_tune <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
  set_engine("kknn") |>
  set_mode("classification")

# k-values
age_ks <- tibble(neighbors = 1:20)

In [ ]:
# v-fold
age_vfold <- vfold_cv(age_train, v = 5, strata = subscribe)

In [ ]:
# workflow with validation
age_tune_fit <- workflow() |>
  add_recipe(age_recipe) |>
  add_model(age_tune) |>
  tune_grid(resamples = age_vfold, grid = players_ks) 

In [ ]:
# accuracy
age_tune_accuracy <- age_tune_fit |> collect_metrics() |>
  filter(.metric == "accuracy") |>
  select(neighbors, mean) |>
  arrange(-mean)
head(age_tune_accuracy)

In [ ]:
age_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = 8) |>
  set_engine("kknn") |>
  set_mode("classification")

In [ ]:
age_vfold <- vfold_cv(age_train, v = 5, strata = subscribe)

In [ ]:
age_fit <- workflow() |>
  add_recipe(age_recipe) |>
  add_model(age_spec) |>
  fit(data = age_train)

age_test_predictions <- predict(age_fit, age_test) |>
  bind_cols(age_test)
head(age_test_pred)

In [ ]:
age_test_predictions|>
  metrics(truth = subscribe, estimate = .pred_class) |>
 filter(.metric == "accuracy")

In [ ]:
confusion_matrix_result <- confusion_matrix_result <- confusionMatrix(
  data      = age_test_predictions$.pred_class,   
  reference = age_test_predictions$subscribe,     
  positive  = "TRUE")
confusion_matrix_result

## Experience (ordinal encoding)

In [ ]:
experience_encoded = factor(clean_players$experience,
  levels = c("Beginner", "Amateur", "Regular", "Veteran", "Pro"),
  ordered = TRUE)
experience_numeric = as.numeric(experience_encoded)

In [ ]:
clean_experience_players <- clean_players |> mutate( experience_numeric = experience_numeric)
head(clean_experience_players)

In [ ]:
experience_split <- initial_split(clean_experience_players, prop = 0.75, strata = subscribe)
experience_train <- training(experience_split)
experience_test <- testing(experience_split)

experience_recipe <- recipe(subscribe ~ experience_numeric, data = clean_experience_players) |>
step_dummy(all_nominal(), -all_outcomes()) |>  
  step_scale(all_numeric(), -all_outcomes())

experience_tune <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
  set_engine("kknn") |>
  set_mode("classification")

experience_ks <- tibble(neighbors = 1:20)

In [ ]:
experience_vfold <- vfold_cv(experience_train, v = 5, strata = subscribe)

# workflow with validation
experience_tune_fit <- workflow() |>
  add_recipe(experience_recipe) |>
  add_model(experience_tune) |>
  tune_grid(resamples = experience_vfold, grid = experience_ks) 
 
# accuracy
experience_tune_accuracy <- experience_tune_fit |> collect_metrics() |>
  filter(.metric == "accuracy") |>
  select(neighbors, mean) |>
  arrange(-mean)
head(experience_tune_accuracy)

In [ ]:
experience_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = 19) |>
  set_engine("kknn") |>
  set_mode("classification")

In [ ]:
experience_vfold <- vfold_cv(experience_train, v = 5, strata = subscribe)

In [ ]:
experience_fit <- workflow() |>
  add_recipe(experience_recipe) |>
  add_model(experience_spec) |>
  fit(data = age_train)

## Discussion

summarize what you found(accuracy)
discuss whether this is what you expected to find
discuss what impact could such findings have
discuss what future questions could this lead to

## Reference 